# Tool Behavior Analysis

This notebook studies the estimator tools directly before we ask the agent to choose among them.

That separation is useful for learning. If the agent later makes a bad choice, we want to know whether the failure came from:

- the estimator itself being weak in that setting
- the agent choosing the wrong estimator
- the agent interpreting diagnostics badly


## Step 1: Load Campaigns And Tool Functions

This cell loads the materialized campaign dataset from disk and imports the three estimator tools.

Why this matters:

- using the saved dataset keeps the comparison reproducible
- importing the tools directly lets us inspect their statistical behavior without any agent loop in the way


In [ ]:
from iroas_agent.data import DEFAULT_DATASET_PATH, load_campaign_dataset
from iroas_agent.tools import geo_diff_in_diff_tool, observational_estimator_tool, rct_estimator_tool

In [ ]:
campaigns = load_campaign_dataset(DEFAULT_DATASET_PATH)
len(campaigns)

## Step 2: Run Each Estimator Where It Is Allowed

This cell loops through the campaigns and applies each tool only when its required data is available.

What the code is doing:

- uses RCT estimates only on campaigns with lift-test data
- uses geo estimates only on campaigns with geo data
- uses observational estimates on every campaign because it is the fallback tool

Why this matters:

- it mirrors the constraints the agent faces later
- it makes the bias-variance tradeoffs visible without any prompting effects


In [ ]:
records = []
for campaign in campaigns:
    truth = campaign.hidden.true_iroas
    if campaign.observed.has_lift_test:
        records.append({'tool': 'rct', 'predicted': rct_estimator_tool(campaign)['estimated_iROAS'], 'true': truth})
    if campaign.observed.has_geo_experiment:
        records.append({'tool': 'geo', 'predicted': geo_diff_in_diff_tool(campaign)['estimated_iROAS'], 'true': truth})
    records.append({'tool': 'observational', 'predicted': observational_estimator_tool(campaign)['estimated_iROAS'], 'true': truth})

## Step 3: Summarize Error By Tool Type

This cell computes simple error summaries.

What to look for:

- mean error as a rough measure of bias
- standard deviation as a rough measure of variability
- count so you know how much evidence supports each estimate of behavior

Why this matters:

- the ReAct agent should prefer stronger causal evidence, but it should also be aware of reliability and noise
- seeing these summaries makes the agent's future choices easier to interpret


In [ ]:
import pandas as pd

df = pd.DataFrame(records)
summary = df.assign(error=lambda x: x['predicted'] - x['true']).groupby('tool')['error'].agg(['mean', 'std', 'count'])
summary

## Step 4: Visualize Predicted Versus True iROAS

The scatterplot below helps you see how each tool behaves relative to the ideal diagonal line.

Why this matters:

- points tightly around the diagonal suggest stronger calibration
- systematic offsets from the diagonal suggest bias
- wide spread around the diagonal suggests higher variance


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)
for ax, tool_name in zip(axes, ['rct', 'geo', 'observational']):
    subset = df[df['tool'] == tool_name]
    ax.scatter(subset['true'], subset['predicted'], alpha=0.6)
    ax.plot([subset['true'].min(), subset['true'].max()], [subset['true'].min(), subset['true'].max()])
    ax.set_title(tool_name)
    ax.set_xlabel('true iROAS')
axes[0].set_ylabel('predicted iROAS')
plt.tight_layout()

## What To Take Away

The point of this notebook is not to crown a universally best estimator.

Instead, the takeaway is that each tool has a profile:

- RCT is usually the strongest causal signal when available
- geo is useful when RCT is missing but is not as clean
- observational is always available but can be confidently wrong

That is exactly the decision landscape the agent must navigate.